# Lab 4. Kafka

`kafka-topics.sh` - zamiast scieżki do tranzakcji na prywatnym środowisku <br>
`kafka-topics.sh --create --topic [topic_name] boostrap-serverver broker:9092` <br>
`kafka-topics.sh --list boostrap-server broker:9092`

In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Writing producer.py


In [3]:
import os
import pyspark

# Autodetekcja wersji Sparka → właściwy connector
spark_version = pyspark.__version__
print(f"Wykryta wersja PySpark: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2" # Wersja dla prywatnego środowiska
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Użyty connector:        {KAFKA_PACKAGE}")

# Musi być ustawione PRZED SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell' # Zmienna środowiskowa

Wykryta wersja PySpark: 4.0.0.dev2
Użyty connector:        org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE) # Musi być wczytywane z tym configiem
    .getOrCreate()                                # jeżeli korzystamy z notebooka
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

query = (kafka_raw.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---+-----+-----+---------+------+---------+-------------+
|key|value|topic|partition|offset|timestamp|timestampType|
+---+-----+-----+---------+------+---------+-------------+
+---+-----+-----+---------+------+---------+-------------+

Batch ID: 1
+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+-------------+
|key |value                                                                                                                                                                        

Kafka wysyła coś co ospark interpretuje jako ramke (dane binarne), nas będzie interesować `value` i `timestamp`

Restartujemy kernel i uruchamiamy jedną komórkę.

In [3]:
from pyspark.sql import SparkSession
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
query = (kafka_raw.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+---+-----+-----+---------+------+---------+-------------+
|key|value|topic|partition|offset|timestamp|timestampType|
+---+-----+-----+---------+------+---------+-------------+
+---+-----+-----+---------+------+---------+-------------+



Znowu restartujemy kernel.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = kafka_raw.select("value")
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+
|value|
+-----+
+-----+

Batch ID: 1
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## Wersja skryptowa

In [1]:
%%file test.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = kafka_raw.select("value")
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    #.option("truncate", False) 
    .start()
)
 
query.awaitTermination()

Writing test.py


`spark-submit test.py` - uruchomienie w terminalu

Uruchomenie wersji skryptowej: <br>
`spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2 test.py`

## Faza string cast

Restart kernela

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = ( kafka_raw.select(
        from_json(col("value").cast("string"), tx_schema).alias("tx")
     )
)

df2 = (
    df.select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp"))
)
 
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+---+
|tx |
+---+
+---+

Batch ID: 1
+----------------------------------------------------------------------+
|tx                                                                    |
+----------------------------------------------------------------------+
|{TX4095, u17, 2177.41, Gdańsk, żywność, 2026-04-27T10:51:37.800627}   |
|{TX1897, u13, 12.72, Wrocław, elektronika, 2026-04-27T10:51:38.802851}|
+----------------------------------------------------------------------+

Batch ID: 2
+------------------------------------------------------------------------+
|tx                                                                      |
+------------------------------------------------------------------------+
|{TX3073, u13, 4441.17, Wrocław, żywność, 2026-04-27T10:51:39.805118}    |
|{TX1040, u13, 3297.67, Wrocław, elektronika, 2026-04-27T10:51:40.807094}|
|{TX1145, u07, 311.86, Kraków, elektronika, 2026-04-27T10:51:41.808295}  |
+-------------------------------------------------

## Faza 3

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)
 
df = ( kafka_raw.select(
        from_json(col("value").cast("string"), tx_schema).alias("tx")
     )
)
 
df2 = ( df.select("tx.*")
      .withColumn("timestamp", to_timestamp("timestamp"))
      )
 
query = (df2.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+-------+------+-----+--------+---------+
|tx_id|user_id|amount|store|category|timestamp|
+-----+-------+------+-----+--------+---------+
+-----+-------+------+-----+--------+---------+

Batch ID: 1
+------+-------+------+-------+--------+--------------------------+
|tx_id |user_id|amount|store  |category|timestamp                 |
+------+-------+------+-------+--------+--------------------------+
|TX3139|u10    |6.43  |Wrocław|żywność |2026-04-27 10:54:11.018015|
+------+-------+------+-------+--------+--------------------------+

Batch ID: 2
+------+-------+-------+--------+--------+--------------------------+
|tx_id |user_id|amount |store   |category|timestamp                 |
+------+-------+-------+--------+--------+--------------------------+
|TX3149|u09    |2256.04|Gdańsk  |odzież  |2026-04-27 10:54:12.026577|
|TX6943|u19    |1562.52|Wrocław |książki |2026-04-27 10:54:13.027844|
|TX5277|u02    |2404.82|Warszawa|książki |2026-04-27 10:54:14.030573|
+------+--

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

parsed = (
    kafka_raw.select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*").withColumn("timestamp", to_timestamp("timestamp"))
)

 
query = (parsed.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+-------+------+-----+--------+---------+
|tx_id|user_id|amount|store|category|timestamp|
+-----+-------+------+-----+--------+---------+
+-----+-------+------+-----+--------+---------+

Batch ID: 1
+------+-------+-------+--------+--------+--------------------------+
|tx_id |user_id|amount |store   |category|timestamp                 |
+------+-------+-------+--------+--------+--------------------------+
|TX4931|u11    |4065.19|Warszawa|żywność |2026-04-27 10:58:40.514736|
|TX7298|u01    |899.89 |Warszawa|żywność |2026-04-27 10:58:41.516924|
|TX9079|u14    |2588.88|Wrocław |żywność |2026-04-27 10:58:42.518908|
+------+-------+-------+--------+--------+--------------------------+

Batch ID: 2
+------+-------+-------+------+--------+--------------------------+
|tx_id |user_id|amount |store |category|timestamp                 |
+------+-------+-------+------+--------+--------------------------+
|TX9740|u15    |3021.77|Kraków|książki |2026-04-27 10:58:43.521124|
|TX8404|

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import window, count, sum as _sum, round as _round
 
 
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
 
spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
 
def process_batch(df, batch_id, tstop=40):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
 
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])
 
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

parsed = (
    kafka_raw.select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*").withColumn("timestamp", to_timestamp("timestamp"))
)

windowed = (
    parsed
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
)
 
query = (parsed.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    #.option("truncate", False) 
    .start()
)

Batch ID: 0
+-----+-------+------+-----+--------+---------+
|tx_id|user_id|amount|store|category|timestamp|
+-----+-------+------+-----+--------+---------+
+-----+-------+------+-----+--------+---------+

Batch ID: 1
+------+-------+-------+--------+-----------+--------------------------+
|tx_id |user_id|amount |store   |category   |timestamp                 |
+------+-------+-------+--------+-----------+--------------------------+
|TX7320|u12    |1807.92|Warszawa|elektronika|2026-04-27 11:08:54.783059|
|TX8424|u16    |969.3  |Warszawa|elektronika|2026-04-27 11:08:55.784013|
+------+-------+-------+--------+-----------+--------------------------+

Batch ID: 2
+------+-------+-------+-------+-----------+--------------------------+
|tx_id |user_id|amount |store  |category   |timestamp                 |
+------+-------+-------+-------+-----------+--------------------------+
|TX9354|u14    |1634.43|Gdańsk |elektronika|2026-04-27 11:08:56.786289|
|TX4116|u12    |3394.15|Kraków |elektronika|